In [ ]:
import h5py
import swiftsimio as sw
import numpy as np
from unyt import unyt_array, Mpc
import pandas as pd

# --- Config ---
snapshot_path = "/cosma8/data/dp004/flamingo/Runs/L1000N1800/HYDRO_FIDUCIAL/snapshots/flamingo_0077/flamingo_0077.hdf5"
catalog_path = "/cosma8/data/dp004/flamingo/Runs/L1000N1800/HYDRO_FIDUCIAL/SOAP/halo_properties_0077.hdf5"
redshift = 0
temperature_cut = 1e5  # K

# Load catalog and select massive clusters
cat = h5py.File(catalog_path, "r")
massive_mask = cat["SO/500_crit/TotalMass"][:] > 10**14.5
cluster_indices = np.where(massive_mask)[0]

# Store results
gas_masses_all = []
gas_masses_hot = []
tsl_all_list = []
tsl_hot_list = []

count = 0
for idx in cluster_indices:
    # Cluster centre and radius
    xc = unyt_array(cat["VR/CentreOfPotential"][:, 0], Mpc)[idx] / (1 + redshift)
    yc = unyt_array(cat["VR/CentreOfPotential"][:, 1], Mpc)[idx] / (1 + redshift)
    zc = unyt_array(cat["VR/CentreOfPotential"][:, 2], Mpc)[idx] / (1 + redshift)
    r500c = unyt_array(cat["SO/500_crit/SORadius"][:], Mpc)[idx]

    # Define comoving load region
    load_region = unyt_array(
        [[xc - r500c, xc + r500c],
         [yc - r500c, yc + r500c],
         [zc - r500c, zc + r500c]], Mpc)
    comoving_region = load_region * (1 + redshift)

    # Load snapshot region
    mask = sw.mask(snapshot_path)
    mask.constrain_spatial(comoving_region)
    data = sw.load(snapshot_path, mask=mask)

    coords = data.gas.coordinates.to(Mpc) / (1 + redshift)
    dx = coords[:, 0] - xc
    dy = coords[:, 1] - yc
    dz = coords[:, 2] - zc
    r = np.sqrt(dx**2 + dy**2 + dz**2)

    # Gas mass in full sphere r < r500c
    full_mask = r <= r500c
    m_full = data.gas.masses[full_mask]
    T_full = data.gas.temperatures[full_mask]

    gas_masses_all.append(m_full.sum().to_value("Msun"))
    gas_masses_hot.append(m_full[T_full > temperature_cut].sum().to_value("Msun"))

    # Tsl in 0.15 < r/r500c <= 1
    shell_mask = (r > 0.15 * r500c) & (r <= r500c)
    T_shell = data.gas.temperatures[shell_mask]
    rho_shell = data.gas.densities[shell_mask]
    m_shell = data.gas.masses[shell_mask]

    if len(T_shell) > 0:
        w_all = m_shell * rho_shell / T_shell**0.75
        tsl_all = (w_all * T_shell).sum() / w_all.sum()
        tsl_all_list.append(tsl_all.to_value("K"))
    else:
        tsl_all_list.append(np.nan)

    # Hot gas Tsl in shell
    hot_mask = T_shell > temperature_cut
    if hot_mask.sum() > 0:
        T_hot = T_shell[hot_mask]
        m_hot = m_shell[hot_mask]
        rho_hot = rho_shell[hot_mask]
        w_hot = m_hot * rho_hot / T_hot**0.75
        tsl_hot = (w_hot * T_hot).sum() / w_hot.sum()
        tsl_hot_list.append(tsl_hot.to_value("K"))
    else:
        tsl_hot_list.append(np.nan)

    # Count
    count += 1
    print(count)
    if count >= 15:
        break

# Save to dataframe
df = pd.DataFrame({
    "GasMass_All": gas_masses_all,
    "GasMass_Hot": gas_masses_hot,
    "Tsl_All": tsl_all_list / 1.16 * 10**7,
    "Tsl_Hot": tsl_hot_list / 1.16 * 10**7
})

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15


In [19]:
df['Tsl_All'] /= 1.16 * 10**7
df['Tsl_Hot'] /= 1.16 * 10**7

In [20]:
df

,GasMass_All,GasMass_Hot,Tsl_All,Tsl_Hot
0,49658223000000.0,49363133000000.0,0.011,4.337594
1,53847393000000.0,53468446000000.0,0.004124,4.628526
2,44513712000000.0,43949020000000.0,0.002052,4.116708
3,35852256000000.0,35674180000000.0,0.005638,3.686891
4,39145477000000.0,38825070000000.0,0.003517,4.037676
5,58873450000000.0,58255498000000.0,0.002713,4.711664
6,74377110000000.0,73803020000000.0,0.003554,5.435582
7,38170075000000.0,37852090000000.0,0.003016,3.73169
8,43953466000000.0,43377200000000.0,0.001954,2.973867
9,53473207000000.0,53095036000000.0,0.003414,4.20887


In [ ]:
# df.to_csv("gas_mass_Tsl_comparison.csv", index=False)